# Karşılaştırma — Tüm Yöntemler

10 notebook'u çalıştırdıktan sonra bu notebook `results/_index.csv` üzerinden tüm yöntemleri karşılaştırma grafiklerine döker. Sonuçlar `results/_compare/` altına yazılır.

In [ ]:
# Setup
import os, sys
NB_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
sys.path.insert(0, NB_DIR)
import importlib, arf_lib; importlib.reload(arf_lib)
print('arf_lib:', arf_lib.__file__)

In [ ]:
# Konfig
RESULTS_ROOT = './results'         # 10 notebook'un yazdığı yer
OUTPUT_DIR   = './results/_compare' # karşılaştırma çıktıları
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# _index.csv'yi yükle
import pandas as pd
from pathlib import Path
idx_path = Path(RESULTS_ROOT) / '_index.csv'
if not idx_path.exists():
    raise FileNotFoundError(f'Run yok: {idx_path}. Önce diğer notebook\'ları çalıştır.')
df = pd.read_csv(idx_path)
print(f'{len(df)} run yüklendi. Yöntemler: {sorted(df.method.unique())}')
df.head()

In [ ]:
# Her yöntem için en iyi (min PPL) ve mean ± std
summary = (df.groupby('method')
             .agg(ppl_mean=('test_ppl','mean'),
                  ppl_std =('test_ppl','std'),
                  ppl_min =('test_ppl','min'),
                  bpc_mean=('test_bpc','mean'),
                  duration_mean=('duration_s','mean'),
                  params_mean=('params_total','mean'),
                  n_runs  =('test_ppl','count'))
             .reset_index()
             .sort_values('ppl_mean'))
summary['ppl_std'] = summary['ppl_std'].fillna(0)
summary.to_csv(f'{OUTPUT_DIR}/summary.csv', index=False)
summary

In [ ]:
# Karşılaştırma grafikleri
import matplotlib.pyplot as plt
import numpy as np

PALETTE = {
    'baseline':  '#6b7280',
    'A1_PCR':    '#3b82f6',
    'A2_LI_SCR': '#1d4ed8',
    'A3_RBR':    '#0ea5e9',
    'B1_MI_S3T': '#22c55e',
    'B2_CMA':    '#15803d',
    'B3_FWML':   '#84cc16',
    'C1_PCUN':   '#f97316',
    'C2_EP_T':   '#ea580c',
    'C3_CHA':    '#dc2626',
}

# 1) PPL bar (mean ± std)
fig, ax = plt.subplots(figsize=(11, 6))
s = summary.copy()
colors = [PALETTE.get(m, '#888888') for m in s['method']]
ax.bar(s['method'], s['ppl_mean'], yerr=s['ppl_std'], capsize=4, color=colors, edgecolor='black')
if 'baseline' in s['method'].values:
    base = s.loc[s['method']=='baseline','ppl_mean'].iloc[0]
    ax.axhline(base, color='red', linestyle='--', alpha=0.6, label='baseline')
    ax.legend()
ax.set_ylabel('Test PPL (mean ± std)')
ax.set_title('Method comparison — Test perplexity')
plt.xticks(rotation=30, ha='right'); plt.grid(axis='y', alpha=0.3)
plt.savefig(f'{OUTPUT_DIR}/ppl_compare.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 2) Speed–accuracy scatter
fig, ax = plt.subplots(figsize=(10, 6))
for _, r in summary.iterrows():
    ax.scatter(r['duration_mean'], r['ppl_mean'], color=PALETTE.get(r['method'],'#888'), s=140, edgecolors='black')
    ax.annotate(r['method'], (r['duration_mean'], r['ppl_mean']), fontsize=8, xytext=(5,5), textcoords='offset points')
ax.set_xlabel('Training time (s)'); ax.set_ylabel('Test PPL')
ax.set_title('Speed vs. accuracy'); ax.grid(alpha=0.3)
plt.savefig(f'{OUTPUT_DIR}/speed_accuracy.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 3) Parameter count
fig, ax = plt.subplots(figsize=(11, 5))
s2 = summary.sort_values('params_mean')
colors2 = [PALETTE.get(m,'#888') for m in s2['method']]
ax.bar(s2['method'], s2['params_mean']/1e6, color=colors2, edgecolor='black')
ax.set_ylabel('Total parameters (M)'); ax.set_title('Parameter count by method')
plt.xticks(rotation=30, ha='right'); plt.grid(axis='y', alpha=0.3)
plt.savefig(f'{OUTPUT_DIR}/param_count.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# 4) Seed varyansı (multi-seed run varsa)
multi = df.groupby('method').filter(lambda x: len(x) >= 2)
if not multi.empty:
    fig, ax = plt.subplots(figsize=(11, 6))
    groups = list(multi.groupby('method'))
    data = [g['test_ppl'].values for _, g in groups]
    labels = [m for m,_ in groups]
    bp = ax.boxplot(data, labels=labels, patch_artist=True)
    for patch, m in zip(bp['boxes'], labels):
        patch.set_facecolor(PALETTE.get(m,'#888')); patch.set_alpha(0.6)
    ax.set_ylabel('Test PPL'); ax.set_title('Seed variance per method')
    plt.xticks(rotation=30, ha='right'); plt.grid(axis='y', alpha=0.3)
    plt.savefig(f'{OUTPUT_DIR}/seed_variance.png', dpi=150, bbox_inches='tight'); plt.show()
else:
    print('Multi-seed run yok — seed varyansı atlanıyor.')

In [ ]:
# Özet
print('Sonuç dizini:', OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print('  •', f)